## Stratified Train–Validation–Test Split

To ensure that the training, validation, and test sets are representative of the original dataset distribution, a **stratified split** was performed.

The stratification was based on two characteristics of the original noisy input text:

1. **Sentence length**
   - Sentences were grouped into length-based categories to preserve the distribution of short and long sentences across all subsets.

2. **Punctuation presence**
   - The dataset was divided into:
     - Sentences containing punctuation
     - Sentences without punctuation

A combined stratification label was created using both features, ensuring that each subset maintains a similar proportion of sentence lengths and punctuation patterns.

### Split Ratio

The dataset was divided into three subsets:

| Dataset | Percentage |
|---------|------------|
| Training Set | 80% |
| Validation Set | 10% |
| Test Set | 10% |

### Verification

After splitting, the distributions of:

- Sentence length categories
- Punctuation presence

were compared across the training, validation, and test sets to confirm that the stratification preserved the original dataset characteristics.

This approach ensures a fair evaluation by preventing the test set from containing a disproportionate number of shorter/longer sentences or punctuated/unpunctuated examples.

**Note:** Only characteristics of the original input text were used during stratification. No information from the target standardized text was used, preventing data leakage.

In [ ]:
import os
import json
import string
import pandas as pd

from sklearn.model_selection import train_test_split

INPUT_FILE = (
    "C:/Users/ae/OneDrive/thesis/data/data_versions/"
    "mc_joint_text_restoration.json"
)

OUTPUT_DIR = "../data/dataset_split"

TRAIN_FILE = os.path.join(OUTPUT_DIR, "train.json")
VAL_FILE   = os.path.join(OUTPUT_DIR, "validation.json")
TEST_FILE  = os.path.join(OUTPUT_DIR, "test.json")

RANDOM_SEED = 18

# Create output directory if it does not exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load the dataset
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)

print(f"Total samples: {len(df)}")

# ============================================================
# CREATE SENTENCE LENGTH FEATURE
# ============================================================

# Compute character-level sentence length
df["text_length"] = df["text"].apply(len)

# Divide sentences into four length categories based on dataset quartiles
df["length_bucket"] = pd.qcut(
    df["text_length"],
    q=4,
    labels=[
        "short",
        "medium",
        "long",
        "very_long"
    ],
    duplicates="drop"
)

# ============================================================
# CREATE PUNCTUATION FEATURE
# ============================================================

punctuation_chars = set(string.punctuation)

def has_punctuation(text):
    """
    Detect whether a sentence contains
    at least one punctuation character.
    """
    return any(char in punctuation_chars for char in text)


df["punctuation_status"] = df["text"].apply(
    lambda x: (
        "with_punctuation"
        if has_punctuation(x)
        else "without_punctuation"
    )
)



# ============================================================
# 6. CREATE STRATIFICATION LABEL
# ============================================================

# Combine sentence length and punctuation status
# to preserve both distributions during splitting

df["stratify_label"] = (
    df["length_bucket"].astype(str)
    + "_"
    + df["punctuation_status"]
)


print("\nStratification distribution:")
print(df["stratify_label"].value_counts())



# ============================================================
# 7. CREATE TRAIN / VALIDATION / TEST SPLITS
# ============================================================

# First split:
# Training = 80%
# Temporary set = 20%

train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=RANDOM_SEED,
    stratify=df["stratify_label"]
)


# Second split:
# Validation = 10%
# Test = 10%

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=RANDOM_SEED,
    stratify=temp_df["stratify_label"]
)



# ============================================================
# 8. REMOVE TEMPORARY FEATURES
# ============================================================

helper_columns = [
    "text_length",
    "length_bucket",
    "punctuation_status",
    "stratify_label"
]


train_df = train_df.drop(columns=helper_columns)
val_df = val_df.drop(columns=helper_columns)
test_df = test_df.drop(columns=helper_columns)



# ============================================================
# 9. SAVE SPLIT DATASETS
# ============================================================

train_df.to_json(
    TRAIN_FILE,
    orient="records",
    force_ascii=False,
    indent=4
)

val_df.to_json(
    VAL_FILE,
    orient="records",
    force_ascii=False,
    indent=4
)

test_df.to_json(
    TEST_FILE,
    orient="records",
    force_ascii=False,
    indent=4
)


print("\nDataset splits saved successfully:")
print(TRAIN_FILE)
print(VAL_FILE)
print(TEST_FILE)



# ============================================================
# 10. VERIFY SPLIT DISTRIBUTIONS
# ============================================================

def check_distribution(name, dataframe):
    """
    Display sentence length and punctuation
    distributions after splitting.
    """

    temp = dataframe.copy()

    temp["length_bucket"] = pd.qcut(
        temp["text"].apply(len),
        q=4,
        labels=[
            "short",
            "medium",
            "long",
            "very_long"
        ],
        duplicates="drop"
    )

    temp["punctuation_status"] = temp["text"].apply(
        lambda x: (
            "with_punctuation"
            if has_punctuation(x)
            else "without_punctuation"
        )
    )

    print("\n" + "=" * 50)
    print(name)
    print("=" * 50)

    print(f"Number of samples: {len(temp)}")

    print("\nPunctuation distribution (%)")
    print(
        temp["punctuation_status"]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
    )

    print("\nSentence length distribution (%)")
    print(
        temp["length_bucket"]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
    )



check_distribution("TRAIN SET", train_df)
check_distribution("VALIDATION SET", val_df)
check_distribution("TEST SET", test_df)

Total samples: 13830

Stratification distribution:
stratify_label
short_without_punctuation        2307
very_long_with_punctuation       2218
long_with_punctuation            1878
medium_without_punctuation       1775
medium_with_punctuation          1625
long_without_punctuation         1518
short_with_punctuation           1275
very_long_without_punctuation    1234
Name: count, dtype: int64

Dataset splits saved successfully:
../data/dataset_split\train.json
../data/dataset_split\validation.json
../data/dataset_split\test.json

TRAIN SET
Number of samples: 11064

Punctuation distribution (%)
punctuation_status
with_punctuation       50.58
without_punctuation    49.42
Name: proportion, dtype: float64

Sentence length distribution (%)
length_bucket
short        25.90
very_long    24.95
medium       24.58
long         24.56
Name: proportion, dtype: float64

VALIDATION SET
Number of samples: 1383

Punctuation distribution (%)
punctuation_status
with_punctuation       50.61
without_punctu